# E-Commerce Behaviour Pipeline
## Big Data Project — Group A7

Multi-stage distributed data pipeline processing **~14.6 GB** of e-commerce event logs using **Dask** and **Apache Spark**.

| | |
|---|---|
| **Dataset** | [Kaggle — eCommerce behavior data from multi-category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) |
| **Files** | `2019-Oct.csv` (5.6 GB) · `2019-Nov.csv` (9.0 GB) |
| **Schema** | `event_time`, `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session` |

### Pipeline Stages
1. **Ingestion & Validation** — load CSVs, schema enforcement, null / range checks, cleaning
2. **Dask Processing** — `filter`, `aggregate`, `pivot`, `assign`, advanced `groupby` (5 analyses)
3. **Spark Processing** — same questions via PySpark + window function, reads **raw CSV directly**
4. **Storage** — intermediate + final results written as **Parquet**
5. **Framework Comparison** — timing and ergonomics summary

In [15]:
import os
os.environ['TEST_MODE'] = 'True'


In [16]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')   # make src/ importable from notebooks/

import dask
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from pathlib import Path
import pandas as pd

from src.config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, LOGS_DIR,
    RAW_OCT, RAW_NOV,
    PARQUET_VALIDATED, RESULTS_DIR,
    VALID_EVENT_TYPES, DTYPES,
)
from src.logger     import StructuredLogger
from src.validation import validate_schema, data_quality_report, clean_data
from src.ingestion  import load_csv, load_csvs
from src.transformations_dask import (
    compute_revenue_by_category,
    compute_conversion_funnel,
    compute_hourly_activity,
    compute_session_stats,
    compute_top_brands,
)
from src.storage import save_parquet_dask, save_parquet_pandas, load_parquet

try:
    from pyspark.sql import SparkSession
    from src.transformations_spark import (
        get_spark_session, load_csv_spark,
        compute_revenue_by_category_spark,
        compute_conversion_funnel_spark,
        compute_window_rank_spark,
        compute_hourly_activity_spark,
    )
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    print('PySpark not found — Spark stage will be skipped.')

log = StructuredLogger('pipeline')
timings: dict[str, float] = {}

for d in [OUTPUT_DIR, LOGS_DIR, PARQUET_VALIDATED, RESULTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Spark        : {SPARK_AVAILABLE}')

Project root : /home/albin/Skola/Big Data/Project/github/Group-A7-Project
Data dir     : /home/albin/Skola/Big Data/Project/github/Group-A7-Project/data
Output dir   : /home/albin/Skola/Big Data/Project/github/Group-A7-Project/output
Spark        : True


---
## Stage 1 — Data Ingestion & Validation

**Operations:** load two CSVs → schema check → null / value-range assertions → row-level cleaning  
**Error handling:** `on_bad_lines='skip'` drops malformed CSV rows; retry logic handles transient I/O failures; `RuntimeError` raised on schema mismatch.

> **Note:** `load_csvs()` passes both file paths directly to `dd.read_csv` as a list — Dask handles this natively without any concat or merge pass. All computation is **lazy** until `save_parquet_dask()` triggers the first full read.

In [17]:
log.info('pipeline_started')

# ── Load both CSVs in one lazy read (no merge/concat pass) ───────────────────
with log.timer('ingestion') as t:
    raw_ddf = load_csvs([RAW_OCT, RAW_NOV])
timings['ingestion'] = t.elapsed

print(f'Partitions : {raw_ddf.npartitions}')
print(f'Columns    : {list(raw_ddf.columns)}')

# ── Schema validation (fast — no compute triggered) ──────────────────────────
schema = validate_schema(raw_ddf)
print(f'\nSchema valid : {schema["valid"]}')
if not schema['valid']:
    raise RuntimeError(f'Missing columns: {schema["missing"]}')

raw_ddf.head(3)

{"ts": "2026-03-13T12:54:58.784236+00:00", "level": "INFO", "event": "pipeline_started"}
{"ts": "2026-03-13T12:54:58.784944+00:00", "level": "INFO", "event": "ingestion_started"}
{"ts": "2026-03-13T12:54:58.785217+00:00", "level": "INFO", "event": "load_csvs_started", "files": ["/home/albin/Skola/Big Data/Project/github/Group-A7-Project/data/2019-Oct.csv", "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/data/2019-Nov.csv"]}
{"ts": "2026-03-13T12:54:58.804638+00:00", "level": "INFO", "event": "load_csvs_ok", "files": 2, "partitions": 114}
{"ts": "2026-03-13T12:54:58.805095+00:00", "level": "INFO", "event": "ingestion_completed", "elapsed_s": 0.02}
{"ts": "2026-03-13T12:54:58.805739+00:00", "level": "INFO", "event": "schema_valid", "columns": ["brand", "category_code", "category_id", "event_time", "event_type", "price", "product_id", "user_id", "user_session"]}


Partitions : 114
Columns    : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Schema valid : True


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,view,44600062,2103807459595387724,<NA>,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00+00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01+00:00,view,17200506,2053013559792632471,furniture.living_room.sofa,<NA>,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8


In [18]:
# ── Clean (lazy — still no compute) ──────────────────────────────────────────
with log.timer('cleaning') as t:
    clean_ddf = clean_data(raw_ddf)
timings['cleaning'] = t.elapsed

# ── Persist validated data as Parquet, partitioned by event_type ──────────────
# This is the first cell that triggers real computation:
#   read 14.6 GB CSV → filter → write Parquet. Expect 10-30 min on a laptop.
print('Writing validated Parquet — this triggers the first full read of the CSV files.')
print('Expected time: 10-30 minutes depending on disk speed.\n')
with log.timer('save_validated') as t:
    save_parquet_dask(clean_ddf, PARQUET_VALIDATED, partition_on=['event_type'])
timings['save_validated'] = t.elapsed

print(f'\nValidated Parquet → {PARQUET_VALIDATED}')
print(f'Elapsed           : {timings["save_validated"]:.1f}s')

# Reload from Parquet — all Dask analyses read this instead of the raw CSVs
validated_ddf = load_parquet(PARQUET_VALIDATED)
print(f'Reloaded partitions: {validated_ddf.npartitions}')

{"ts": "2026-03-13T12:55:01.386502+00:00", "level": "INFO", "event": "cleaning_started"}
{"ts": "2026-03-13T12:55:01.391471+00:00", "level": "INFO", "event": "cleaning_rules_applied"}
{"ts": "2026-03-13T12:55:01.391811+00:00", "level": "INFO", "event": "cleaning_completed", "elapsed_s": 0.01}
{"ts": "2026-03-13T12:55:01.392257+00:00", "level": "INFO", "event": "save_validated_started"}
{"ts": "2026-03-13T12:55:01.392705+00:00", "level": "INFO", "event": "saving_parquet_dask", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/parquet/validated"}


Writing validated Parquet — this triggers the first full read of the CSV files.
Expected time: 10-30 minutes depending on disk speed.



{"ts": "2026-03-13T12:59:26.378842+00:00", "level": "INFO", "event": "saved_parquet_dask", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/parquet/validated"}
{"ts": "2026-03-13T12:59:26.381495+00:00", "level": "INFO", "event": "save_validated_completed", "elapsed_s": 264.99}



Validated Parquet → /home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/parquet/validated
Elapsed           : 265.0s
Reloaded partitions: 332


---
## Stage 2 — Distributed Processing with Dask

Five analyses, each using a distinct operation class:

| # | Analysis | Operations | Scheduler |
|---|---|---|---|
| 1 | Revenue by category | `filter` + `groupby` aggregate | distributed |
| 2 | Conversion funnel | per-event-type `filter` + single-key `groupby` | distributed |
| 3 | Hourly activity | per-event-type `filter` + `assign` + `groupby` | distributed |
| 4 | Session statistics | map-reduce `map_partitions` + pandas re-aggregation | **threaded** (no cluster) |
| 5 | Top brands | `filter` + `nlargest` | distributed |

> **Memory note:** Analysis 4 runs **outside** the distributed cluster because `groupby(user_session)` over millions of unique UUIDs triggers a `repartitiontofewer` step inside Dask's `.compute()` that OOMs workers. Without an active distributed client, the threaded scheduler concatenates partition results directly in the driver process.

In [19]:
import dask

# Allow workers to spill to disk before hitting the kill threshold
dask.config.set({
    'distributed.worker.memory.target':    0.65,
    'distributed.worker.memory.spill':     0.75,
    'distributed.worker.memory.pause':     0.85,
    'distributed.worker.memory.terminate': 0.95,
})

dask_results = {}

# ── Analyses 1, 2, 3, 5 — run inside distributed cluster ────────────────────
with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='4GB') as cluster:
    with Client(cluster) as client:
        print(f'Dashboard : {client.dashboard_link}')
        print(f'Workers   : {len(client.scheduler_info()["workers"])}\n')

        # 1 — Revenue by category
        with log.timer('dask_revenue') as t:
            dask_results['revenue'] = compute_revenue_by_category(validated_ddf)
        timings['dask_revenue'] = t.elapsed
        print('--- Top 10 Categories by Revenue ---')
        print(dask_results['revenue'].head(10).to_string(index=False))

        # 2 — Conversion funnel
        with log.timer('dask_funnel') as t:
            dask_results['funnel'] = compute_conversion_funnel(validated_ddf)
        timings['dask_funnel'] = t.elapsed
        cols = [c for c in ['top_category','view','cart','purchase','purchase_rate','cart_rate']
                if c in dask_results['funnel'].columns]
        print('\n--- Conversion Funnel (top 5 by purchase rate) ---')
        print(dask_results['funnel'].sort_values('purchase_rate', ascending=False).head(5)[cols].to_string(index=False))

        # 3 — Hourly activity
        with log.timer('dask_hourly') as t:
            dask_results['hourly'] = compute_hourly_activity(validated_ddf)
        timings['dask_hourly'] = t.elapsed
        print('\n--- Events by Hour (purchases) ---')
        hourly_purchase = dask_results['hourly'][dask_results['hourly']['event_type'] == 'purchase']
        for _, row in hourly_purchase.iterrows():
            bar = '#' * (int(row['count']) // 50_000)
            print(f'  {int(row["hour"]):2d}:00  {int(row["count"]):>10,}  {bar}')

        # 5 — Top brands
        with log.timer('dask_brands') as t:
            dask_results['brands'] = compute_top_brands(validated_ddf, top_n=20)
        timings['dask_brands'] = t.elapsed
        print('\n--- Top 10 Brands by Purchase Revenue ---')
        print(dask_results['brands'].head(10).to_string(index=False))

# ── Analysis 4: Session stats — no active distributed client ─────────────────
print('\n--- Session Analysis (threaded scheduler, no distributed client) ---')
with log.timer('dask_sessions') as t:
    dask_results['sessions'] = compute_session_stats(validated_ddf)
timings['dask_sessions'] = t.elapsed
sess = dask_results['sessions']
print(f'  Total sessions      : {len(sess):,}')
print(f'  Avg events/session  : {sess["num_events"].mean():.1f}')
print(f'  Median duration     : {sess["session_duration_min"].median():.1f} min')
print(f'  Avg spend/session   : {sess["total_spend"].mean():.2f}')

print(f'\nDask compute total: {sum(v for k,v in timings.items() if k.startswith("dask_")):.1f}s')

{"ts": "2026-03-13T12:59:26.917510+00:00", "level": "INFO", "event": "dask_revenue_started"}


Dashboard : http://127.0.0.1:8787/status
Workers   : 2



2026-03-13 13:59:28,009 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c38fa4d047d0b6605a503224a5b5db82 initialized by task ('shuffle-transfer-c38fa4d047d0b6605a503224a5b5db82', 51) executed on worker tcp://127.0.0.1:33549
2026-03-13 13:59:36,492 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c38fa4d047d0b6605a503224a5b5db82 deactivated due to stimulus 'task-finished-1773406776.4888074'
{"ts": "2026-03-13T12:59:36.530769+00:00", "level": "INFO", "event": "revenue_by_category_done", "categories": 14}
{"ts": "2026-03-13T12:59:36.531280+00:00", "level": "INFO", "event": "dask_revenue_completed", "elapsed_s": 9.61}
{"ts": "2026-03-13T12:59:36.533579+00:00", "level": "INFO", "event": "dask_funnel_started"}


--- Top 10 Categories by Revenue ---
top_category  total_revenue  num_purchases  avg_price
 electronics   381714286.57         916667 416.415434
     unknown    52805443.81         407643 129.538454
  appliances    32223623.59         174022 185.169827
   computers    25373205.34          62332 407.065477
   furniture     4216980.01          19843 212.517261
        auto     2649036.47          21339 124.140610
construction     2013385.72          16500 122.023377
     apparel     1805962.87          22217  81.287432
        kids     1401903.93          11648 120.355763
       sport      718251.75           2725 263.578624


2026-03-13 13:59:37,328 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1dd8c1523e116cf7a09af3d151c140ca initialized by task ('shuffle-transfer-1dd8c1523e116cf7a09af3d151c140ca', 24) executed on worker tcp://127.0.0.1:33549
2026-03-13 14:00:44,014 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1dd8c1523e116cf7a09af3d151c140ca deactivated due to stimulus 'task-finished-1773406844.0139554'
2026-03-13 14:00:44,568 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8cf1df62bb2fbc576e5e7b4cd779d6b5 initialized by task ('shuffle-transfer-8cf1df62bb2fbc576e5e7b4cd779d6b5', 45) executed on worker tcp://127.0.0.1:33549
2026-03-13 14:01:47,433 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8cf1df62bb2fbc576e5e7b4cd779d6b5 deactivated due to stimulus 'task-finished-1773406907.4312012'
2026-03-13 14:01:48,012 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 95ef2d29128c1ed04248ca0c37d36521 initialized by task ('shuffle-transfer-95ef2d29128c

...


--- Top 10 Brands by Purchase Revenue ---
  brand  total_revenue  num_purchases
  apple   238721793.70         308937
samsung   101277413.48         372923
 xiaomi    20453899.25         124908
 huawei     9664104.09          47204
     lg     8626906.72          21606
   acer     6924026.05          13284
lucente     6651658.94          26137
   sony     6341082.98          17038
   oppo     5901500.52          25971
 lenovo     4450744.83          11125


{"ts": "2026-03-13T13:05:19.311297+00:00", "level": "INFO", "event": "dask_sessions_started"}
{"ts": "2026-03-13T13:05:19.311663+00:00", "level": "INFO", "event": "session_stats_phase1_started"}



--- Session Analysis (threaded scheduler, no distributed client) ---


{"ts": "2026-03-13T13:05:33.085085+00:00", "level": "INFO", "event": "session_stats_phase1_done", "partial_rows": 27106776}
{"ts": "2026-03-13T13:05:42.374347+00:00", "level": "INFO", "event": "session_stats_done", "sessions": 23016650}
{"ts": "2026-03-13T13:05:42.383382+00:00", "level": "INFO", "event": "dask_sessions_completed", "elapsed_s": 23.07}


  Total sessions      : 23,016,650
  Avg events/session  : 4.8
  Median duration     : 1.0 min
  Avg spend/session   : 1393.14

Dask compute total: 374.7s


In [20]:
# ── Persist Dask results as Parquet ───────────────────────────────────────────
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_parquet_pandas(dask_results['revenue'],                RESULTS_DIR / 'dask_revenue.parquet')
save_parquet_pandas(dask_results['funnel'],                 RESULTS_DIR / 'dask_funnel.parquet')
save_parquet_pandas(dask_results['hourly'],                 RESULTS_DIR / 'dask_hourly.parquet')
save_parquet_pandas(dask_results['brands'],                 RESULTS_DIR / 'dask_brands.parquet')
save_parquet_pandas(dask_results['sessions'].head(500_000), RESULTS_DIR / 'dask_sessions.parquet')

print('Dask results saved to:', RESULTS_DIR)

{"ts": "2026-03-13T13:05:42.772073+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_revenue.parquet", "rows": 14}
{"ts": "2026-03-13T13:05:42.774586+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_funnel.parquet", "rows": 14}
{"ts": "2026-03-13T13:05:42.776186+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_hourly.parquet", "rows": 72}
{"ts": "2026-03-13T13:05:42.777460+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results/dask_brands.parquet", "rows": 20}
{"ts": "2026-03-13T13:05:42.976073+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "/home/albin/Skola/Big Data/Project/github/Group-A7-Projec

Dask results saved to: /home/albin/Skola/Big Data/Project/github/Group-A7-Project/output/results


---
## Stage 3 — Distributed Processing with Apache Spark

Same analytical questions answered via PySpark for framework comparison.  
Spark reads **directly from the raw CSV files** to avoid Parquet timestamp format incompatibilities between pyarrow (Dask) and the JVM Parquet reader (Spark).  
**Bonus**: analysis 3 uses a **window function** (`RANK() OVER PARTITION BY`) to rank products within each category.

> If PySpark is not installed, this stage is skipped automatically.

In [21]:
spark_results = {}

if not SPARK_AVAILABLE:
    print('Spark not available — skipping Stage 3.')
else:
    from pyspark.sql import functions as F

    spark = get_spark_session('EcomPipeline-A7')
    print(f'Spark version: {spark.version}\n')

    # Spark reads the raw CSVs directly — avoids pyarrow/JVM Parquet incompatibility
    # Use forward-slash paths (Spark on Windows requires this)
    csv_paths = [
        str(RAW_OCT).replace('\\', '/'),
        str(RAW_NOV).replace('\\', '/'),
    ]
    with log.timer('spark_load') as t:
        sdf_raw = load_csv_spark(spark, csv_paths)
    timings['spark_load'] = t.elapsed

    # Apply the same cleaning filters as Stage 1
    valid_events = list(VALID_EVENT_TYPES)
    sdf = (
        sdf_raw
        .dropna(subset=['user_id', 'product_id', 'price', 'event_time', 'user_session'])
        .filter(F.col('event_type').isin(valid_events))
        .filter(F.col('price') >= 0)
        .filter(F.trim(F.col('user_session')) != '')
    )
    print(f'Spark partitions after cleaning: {sdf.rdd.getNumPartitions()}')
    sdf.printSchema()

    # 1 — Revenue by category
    with log.timer('spark_revenue') as t:
        spark_results['revenue'] = compute_revenue_by_category_spark(sdf)
    timings['spark_revenue'] = t.elapsed
    print('--- Spark: Top 10 Categories by Revenue ---')
    spark_results['revenue'].show(10, truncate=False)

    # 2 — Conversion funnel
    with log.timer('spark_funnel') as t:
        spark_results['funnel'] = compute_conversion_funnel_spark(sdf)
    timings['spark_funnel'] = t.elapsed
    print('--- Spark: Conversion Funnel ---')
    spark_results['funnel'].orderBy('purchase_rate', ascending=False).show(10, truncate=False)

    # 3 — Window function: rank products within category
    with log.timer('spark_window') as t:
        spark_results['window'] = compute_window_rank_spark(sdf)
    timings['spark_window'] = t.elapsed
    print('--- Spark: Top Products per Category (window rank) ---')
    spark_results['window'].show(20, truncate=False)

    # 4 — Hourly activity
    with log.timer('spark_hourly') as t:
        spark_results['hourly'] = compute_hourly_activity_spark(sdf)
    timings['spark_hourly'] = t.elapsed

    # Persist Spark results as Parquet
    results_fwd = str(RESULTS_DIR).replace('\\', '/')
    spark_results['revenue'].write.mode('overwrite').parquet(f'{results_fwd}/spark_revenue')
    spark_results['funnel'].write.mode('overwrite').parquet(f'{results_fwd}/spark_funnel')
    spark_results['window'].write.mode('overwrite').parquet(f'{results_fwd}/spark_window_rank')
    spark_results['hourly'].write.mode('overwrite').parquet(f'{results_fwd}/spark_hourly')

    print(f'\nSpark compute total: {sum(v for k,v in timings.items() if k.startswith("spark_")):.1f}s')
    spark.stop()

TypeError: 'JavaPackage' object is not callable

---
## Stage 4 — Results Summary & Framework Comparison

### Storage Strategy

| Layer | Format | Location | Why |
|---|---|---|---|
| Raw input | CSV | `data/` | Source of truth, unchanged |
| Validated intermediate | **Parquet** (partitioned by `event_type`) | `output/parquet/validated/` | Predicate pushdown speeds downstream reads |
| Analysis results | **Parquet** (single file) | `output/results/` | Compact, typed, directly loadable by BI tools |

In [ ]:
# ── Timing comparison ─────────────────────────────────────────────────────────
print('=== Pipeline Stage Timings ===')
print(f'{"Stage":<25} {"Time (s)":>10}')
print('-' * 37)
for stage, elapsed in timings.items():
    print(f'{stage:<25} {elapsed:>10.1f}')
print('=' * 37)
print(f'{"TOTAL":<25} {sum(timings.values()):>10.1f}')

# ── Output files ──────────────────────────────────────────────────────────────
print('\n=== Output Files ===')
for p in sorted(Path(OUTPUT_DIR).rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1_048_576
        print(f'  {str(p.relative_to(PROJECT_ROOT)):<55} {size_mb:>7.2f} MB')

# ── Framework notes ───────────────────────────────────────────────────────────
print("""
=== Framework Comparison Notes ===
Dask
  + Familiar pandas-like API, easy iteration in notebooks
  + Native Python, no JVM overhead
  + Low-latency scheduler startup
  - High-cardinality groupby OOMs via repartitiontofewer in distributed mode;
    must run outside Client context with threaded scheduler
  - Multi-key shuffles require per-key decomposition to avoid OOM

Spark
  + Catalyst optimiser produces efficient physical plans
  + First-class window functions, complex joins
  + Graceful spill-to-disk for large shuffles
  + No format interoperability issues (reads CSV directly)
  - JVM startup adds ~30s latency
  - More verbose API; Windows path handling quirks

Verdict: Dask is faster for simple aggregations on a single node.
Spark is more robust for complex operations (windows, multi-key joins)
and scales better on managed clusters (EMR, Dataproc).
""")